# ID-card OCR results
Run this notebook from `projects/ICP`. It reads CSV files produced by the shared evaluator.

First run the setup and evaluation commands in README. To check the new output format:
```bash
python evaluate.py --group variants --device cpu --limit 5 --output results/smoke-csv
```
Five cards are a smoke check, not a full development or test evaluation. Old JSON output directories are left unchanged.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

RUN_DIR = Path("results/smoke-csv")  # Change to results/development-all or results/test-all later.
run = json.loads((RUN_DIR / "metadata" / "run.json").read_text())
print("Slice:", run["slice"], "| Cards per configuration:", run["cards"], "| Smoke run:", run["limit"] is not None)
results = pd.read_csv(RUN_DIR / "results.csv")
fields = pd.read_csv(RUN_DIR / "fields.csv")
comparisons = pd.read_csv(RUN_DIR / "comparisons.csv")
rankings = pd.read_csv(RUN_DIR / "rankings.csv")

# Keep older completed runs readable after the naming change.
results = results.rename(columns={name: name.replace("historical_", "m1_", 1) for name in results if name.startswith("historical_")})
for table in (results, fields, rankings):
    table["configuration"] = table["configuration"].str.replace(r"^historical_", "", regex=True)
for table in (comparisons, rankings):
    table["group"] = table["group"].replace("historical", "original")
print("Configurations:", len(results))
if run["slice"] != "test":
    print("Development results only. Frozen settings and the full test run are still needed.")


## Accuracy
Higher exact match is better. Lower character error rate (CER) is better. A blank score means its denominator was zero.


In [19]:
accuracy_columns = ["configuration", "cards", "correct_fields", "nonempty_fields", "field_exact_match", "card_exact_match", "mean_field_cer"]
display(results[accuracy_columns].round(4))


,configuration,cards,correct_fields,nonempty_fields,field_exact_match,card_exact_match,mean_field_cer
0,tesseract_default,5,13,40,0.325,0.0,0.5746
1,tesseract_psm6,5,13,40,0.325,0.0,0.5433
2,easyocr_original,5,16,40,0.400,0.0,0.6287
3,easyocr_half,5,13,40,0.325,0.0,0.5798


## Assertions
A1: ID format. A2: ID and birth fields agree. A3: valid birth date. A4: valid gender. `all_pass_rate` means all four passed. Rates range from 0 to 1.


In [20]:
assertion_columns = ["configuration", "A1_pass_rate", "A2_pass_rate", "A3_pass_rate", "A4_pass_rate", "all_pass_rate"]
display(results[assertion_columns].round(4))


,configuration,A1_pass_rate,A2_pass_rate,A3_pass_rate,A4_pass_rate,all_pass_rate
0,tesseract_default,0.8,0.2,0.2,0.4,0.0
1,tesseract_psm6,1.0,0.2,0.2,0.6,0.0
2,easyocr_original,0.6,0.0,0.2,0.4,0.0
3,easyocr_half,0.8,0.0,0.0,0.2,0.0


## Scores for each field
Empty predictions still count as errors for non-empty references.


In [21]:
display(fields.pivot(index="field", columns="configuration", values="exact_match").round(4))
# Raw correct/total counts are available in fields.


configuration,easyocr_half,easyocr_original,tesseract_default,tesseract_psm6
field,,,,
address,0.2,0.4,0.0,0.0
birth_day,0.0,0.2,0.2,0.2
birth_month,0.0,0.0,0.2,0.2
birth_year,0.8,1.0,0.4,0.2
ethnicity,0.2,0.0,0.6,0.6
gender,0.2,0.4,0.4,0.6
id_number,0.8,0.6,0.8,0.8
name,0.4,0.6,0.0,0.0


## Milestone 1 metrics
These preserve the Milestone 1 definitions. Milestone 1 exact match also requires box IoU ≥ 0.3 and uses case folding; required exact match does neither.


In [ ]:
m1_columns = ["configuration"] + [column for column in results if column.startswith("m1_")]
display(results[m1_columns].round(4))


## Do assertions track accuracy?
Start with the all-four pass rate. Positive correlation means the two scores tend to rise together. Blank correlations have a reason, such as every configuration having the same card accuracy. A five-card smoke run is too small to draw conclusions.


In [ ]:
display(comparisons[(comparisons["group"] == "all") & (comparisons["assertion"] == "all")].round(4))
rank_columns = ["configuration", "field_exact_match_rank", "card_exact_match_rank", "all_rank"]
display(rankings.loc[rankings["group"] == "all", rank_columns])
# For T1, also inspect group == "original" after evaluating --group all.


## Inspect predictions
Choose one configuration and field. Text columns are loaded as strings so ID numbers keep their digits and leading zeros. Blank cells are missing predictions, not skipped fields. Matching here uses the same required normalization as evaluation.


In [ ]:
from metrics import normalize

CONFIGURATION = "easyocr_original"
FIELD = "birth_month"  # name, gender, ethnicity, birth_year, birth_month, birth_day, address, id_number
text_columns = [prefix + name for prefix in ("predicted_", "reference_") for name in fields["field"].unique()]
prediction_path = RUN_DIR / f"{CONFIGURATION}.predictions.csv"
if not prediction_path.exists():
    prediction_path = RUN_DIR / f"historical_{CONFIGURATION}.predictions.csv"
predictions = pd.read_csv(prediction_path, dtype={name: "string" for name in text_columns}, keep_default_na=False)
predicted, reference = f"predicted_{FIELD}", f"reference_{FIELD}"
inspection = predictions[["index", predicted, reference, "A1", "A2", "A3", "A4", "all"]].copy()
inspection["field_correct"] = predictions[predicted].map(normalize) == predictions[reference].map(normalize)
display(inspection)


## Run the next evaluation
The notebook delegates evaluation to `evaluate.py`; it does not duplicate scoring. The next cell is commented out to avoid starting a full run by accident. Use a new output directory for each run.


In [ ]:
# import subprocess
# import sys
# subprocess.run([sys.executable, "evaluate.py", "--group", "all", "--device", "cpu",
#                 "--output", "results/development-all"], check=True)
